### Importing libraries

In [11]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve
)

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

import joblib


In [6]:
from models.logistic import get_logistic_model
from models.random_forest import get_random_forest_model
from models.xgboost_model import get_xgboost_model
from models.lightgbm_model import get_lightgbm_model


ModuleNotFoundError: No module named 'models'

### Load datasets

In [12]:
train_df = pd.read_csv("../data/raw/Train.csv")
test_df  = pd.read_csv("../data/raw/Test.csv")

### Separating features and target variable

In [13]:
TARGET = "bank_account"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET].map({"Yes": 1, "No": 0})

### Load the preprocessor

In [14]:
preprocessor = joblib.load("../artifacts/preprocessor.pkl")

AttributeError: Can't get attribute 'binary_encode' on <module '__main__'>

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "Logistic": get_logistic_model(preprocessor),
    "Random Forest": get_random_forest_model(preprocessor),
    "XGBoost": get_xgboost_model(preprocessor, scale_pos_weight),
    "LightGBM": get_lightgbm_model(preprocessor)
}


In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    evaluate_model(model, X_train, y_train, name)


### Fitting the baseline model **Logistic Regression**

In [ ]:
log_reg = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])


### Training the model

In [ ]:
log_reg.fit(X_train, y_train)

### Evaluating the performance

In [ ]:
y_pred = log_reg.predict(X_train)
y_prob = log_reg.predict_proba(X_train)[:, 1]

print("Accuracy:", accuracy_score(y_train, y_pred))
print("ROC AUC:", roc_auc_score(y_train, y_prob))
print(classification_report(y_train, y_pred))


### Fitting the random forest model

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])


In [ ]:
rf_model.fit(X_train, y_train)

#### Evaluating the random forest model

In [ ]:
rf_prob = rf_model.predict_proba(X_train)[:, 1]
print("RF ROC AUC:", roc_auc_score(y_train, rf_prob))


### 3. Fitting XGBoost model

In [ ]:
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        eval_metric="logloss",
        random_state=42
    ))
])


In [ ]:
xgb_model.fit(X_train, y_train)

### 4. Fitting LightGBM model 

In [ ]:
lgbm_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=42
    ))
])


In [ ]:
lgbm_model.fit(X_train, y_train)

### Evaluating models using Precision & Recall

In [ ]:
def evaluate_model(model, X, y, name="Model"):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    print(f"\n{name}")
    print("-" * 40)
    print("Precision:", precision_score(y, y_pred))
    print("Recall:", recall_score(y, y_pred))
    print("F1-score:", f1_score(y, y_pred))
    print("ROC AUC:", roc_auc_score(y, y_prob))


**Evaluating all models**

In [ ]:
evaluate_model(log_reg, X_train, y_train, "Logistic Regression")
evaluate_model(rf_model, X_train, y_train, "Random Forest")
evaluate_model(xgb_model, X_train, y_train, "XGBoost")
evaluate_model(lgbm_model, X_train, y_train, "LightGBM")


In [8]:
import shap
import numpy as np
import pandas as pd


ImportError: Numba needs NumPy 2.3 or less. Got NumPy 2.4.

In [9]:
def get_feature_names(preprocessor):
    feature_names = []

    for name, transformer, cols in preprocessor.transformers_:
        if transformer == "drop":
            continue
        if hasattr(transformer, "get_feature_names_out"):
            feature_names.extend(transformer.get_feature_names_out(cols))
        else:
            feature_names.extend(cols)

    return feature_names


In [10]:
feature_names = get_feature_names(preprocessor)


NameError: name 'preprocessor' is not defined

In [ ]:
X_train_transformed = preprocessor.transform(X_train)


In [ ]:
xgb = xgb_model.named_steps["model"]

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_train_transformed)


In [ ]:
shap.summary_plot(
    shap_values,
    X_train_transformed,
    feature_names=feature_names
)


In [ ]:
shap_df = pd.DataFrame({
    "feature": feature_names,
    "importance": np.abs(shap_values).mean(axis=0)
}).sort_values(by="importance", ascending=False)

shap_df.to_csv("../outputs/feature_importance.csv", index=False)


In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score

def evaluate_model(model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    return {
        "recall": recall_score(y, y_pred),
        "precision": precision_score(y, y_pred),
        "f1": f1_score(y, y_pred),
        "roc_auc": roc_auc_score(y, y_prob)
    }


In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    metrics = evaluate_model(model, X_train, y_train)

    metrics["model"] = name
    results.append(metrics)


In [ ]:
results_df = pd.DataFrame(results).sort_values(
    by="recall", ascending=False
)

results_df


In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]

print("Best model based on recall:", best_model_name)


In [ ]:
import joblib

joblib.dump(best_model, "../artifacts/best_model.pkl")
